In [1]:
#BSWC - Building Systems with ChatGPT API

#Process Inputs: Chaining Prompts

In [2]:
#1.SETTING UP THE OPENAI CLIENT

import os
# import → Keyword used to bring a library into the notebook
# os → Built-in library for interacting with the operating system (e.g., reading environment variables)

from dotenv import load_dotenv
# from → Keyword to import from a specific library
# dotenv → Library (python-dotenv) for reading key-value settings from a .env file
# import → Keyword used to bring in the specified item
# load_dotenv → Function that loads variables from a .env file into the environment

from openai import OpenAI
# from → Keyword to import from a specific library
# openai → Library used to connect to OpenAI's models
# import → Keyword used to bring in the specified item
# OpenAI → Class used to create a client that sends requests to OpenAI's API

load_dotenv(dotenv_path=".env")
# load_dotenv → Function that loads variables from a .env file into the environment
# (dotenv_path=".env") → Argument specifying the path/name of the file to read

openai_api_key = os.getenv("OPENAI_API_KEY")
# openai_api_key → Variable name storing the result
# = → Assignment operator
# os → Built-in operating system library
# .getenv → Function that reads the value of an environment variable
# ("OPENAI_API_KEY") → Argument specifying the name of the variable to read (your secret API key)

client = OpenAI()
# client → Variable name storing the result
# = → Assignment operator
# OpenAI() → Creates the API client object (automatically uses the API key from the environment)

# This cell loads the API key from a .env file and creates the OpenAI client object used by later cells to send requests to the model.

In [3]:
#2.FUNCTION TO GET A COMPLETION FROM A LIST OF MESSAGES

def get_completion_from_messages(messages,
                                  model="gpt-5.6-luna",
                                  temperature=1,
                                  max_tokens=500):
# def → Keyword used to define a function
# get_completion_from_messages → Name given to the function
# (messages, model="gpt-5.6-luna", temperature=1, max_tokens=500) → Parameters: the conversation list, and three settings with default values
# messages → Parameter holding the full list of conversation messages (no default, must be provided)
# model="gpt-5.6-luna" → Parameter setting which model to use, defaulting to gpt-5.6-luna
# temperature=1 → Parameter controlling randomness, defaulting to 1 (gpt-5.6-luna's only supported value)
# max_tokens=500 → Parameter limiting the length of the reply, defaulting to 500 tokens
# : → Marks the start of the function body

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=max_tokens,
    )
    # response → Variable name storing the result
    # = → Assignment operator
    # client → The OpenAI client object created earlier
    # .chat.completions → Attribute giving access to the Chat Completions API
    # .create → Function that sends a request to the model and gets a reply
    # ( → Opens the function's arguments
    # model=model → Argument passing the model name given to the function
    # messages=messages, → Argument passing the conversation list given to the function
    # temperature=temperature, → Argument passing the randomness setting given to the function
    # max_completion_tokens=max_tokens, → Argument passing the output length limit, using the parameter name required by gpt-5.6-luna
    # ) → Closes the function's arguments

    return response.choices[0].message.content
    # return → Keyword that sends a value back from the function
    # response → The reply object returned by the model
    # .choices → Attribute holding a list of possible responses (usually just one)
    # [0] → Indexing operator selecting the first (and only) response
    # .message → Attribute holding the message object of that response
    # .content → Attribute holding the actual answer text

# This function accepts a full messages list (for multi-turn conversations) and lets you customize the model, temperature, and max_tokens, 
# returning just the model's reply text. It uses max_completion_tokens, the parameter name required by gpt-5.6-luna.

In [4]:
# Implement a complex task with multiple prompts
# Extract relevant product and category names

#3.PRODUCT CATEGORY AND NAME EXTRACTION (STRUCTURED OUTPUT PARSING)

delimiter = "####"
# delimiter → Variable name storing the result
# = → Assignment operator
# "####" → String used as a separator to clearly mark the boundaries of the customer's query

system_message = f"""
You will be provided with customer service queries. \
The customer service query will be delimited with \
{delimiter} characters.
Output a python list of objects, where each object has \
the following format:
    'category': <one of Computers and Laptops, \
    Smartphones and Accessories, \
    Televisions and Home Theater Systems, \
    Gaming Consoles and Accessories, 
    Audio Equipment, Cameras and Camcorders>,
OR
    'products': <a list of products that must \
    be found in the allowed products below>

Where the categories and products must be found in \
the customer service query.
If a product is mentioned, it must be associated with \
the correct category in the allowed products list below.
If no products or categories are found, output an \
empty list.

Allowed products: 

Computers and Laptops category:
TechPro Ultrabook
BlueWave Gaming Laptop
PowerLite Convertible
TechPro Desktop
BlueWave Chromebook

Smartphones and Accessories category:
SmartX ProPhone
MobiTech PowerCase
SmartX MiniPhone
MobiTech Wireless Charger
SmartX EarBuds

Televisions and Home Theater Systems category:
CineView 4K TV
SoundMax Home Theater
CineView 8K TV
SoundMax Soundbar
CineView OLED TV

Gaming Consoles and Accessories category:
GameSphere X
ProGamer Controller
GameSphere Y
ProGamer Racing Wheel
GameSphere VR Headset

Audio Equipment category:
AudioPhonic Noise-Canceling Headphones
WaveSound Bluetooth Speaker
AudioPhonic True Wireless Earbuds
WaveSound Soundbar
AudioPhonic Turntable

Cameras and Camcorders category:
FotoSnap DSLR Camera
ActionCam 4K
FotoSnap Mirrorless Camera
ZoomMaster Camcorder
FotoSnap Instant Camera

Only output the list of objects, with nothing else.
"""
# system_message → Variable name storing the result
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }
# You will be provided with customer service queries. → Role/context instruction setting up the extraction task
# The customer service query will be delimited with {delimiter} characters. → Tells the model how the input will be marked off (inserts "####")
# Output a python list of objects, where each object has the following format: → Instruction fixing the required output structure
# 'category': <one of Computers and Laptops, ...> → Defines one possible object type: a category name picked from a fixed list of 6 categories
# OR 'products': <a list of products...> → Defines the other possible object type: a list of specific product names
# Where the categories and products must be found in the customer service query. → Rule restricting output to only what's actually mentioned
# If a product is mentioned, it must be associated with the correct category... → Rule ensuring products are correctly paired with their real category
# If no products or categories are found, output an empty list. → Rule for the no-match case
# Allowed products: → Label introducing the full catalog, grouped by category
# Computers and Laptops category: TechPro Ultrabook, BlueWave Gaming Laptop, PowerLite Convertible, TechPro Desktop, BlueWave Chromebook → First 
#category's 5 valid products

# Smartphones and Accessories category: SmartX ProPhone, MobiTech PowerCase, SmartX MiniPhone, MobiTech Wireless Charger, SmartX EarBuds → Second 
#category's 5 valid products

# Televisions and Home Theater Systems category: CineView 4K TV, SoundMax Home Theater, CineView 8K TV, SoundMax Soundbar, CineView OLED TV → Third 
#category's 5 valid products

# Gaming Consoles and Accessories category: GameSphere X, ProGamer Controller, GameSphere Y, ProGamer Racing Wheel, GameSphere VR Headset → Fourth 
#category's 5 valid products

# Audio Equipment category: AudioPhonic Noise-Canceling Headphones, WaveSound Bluetooth Speaker, AudioPhonic True Wireless Earbuds, WaveSound Soundbar, 
#AudioPhonic Turntable → Fifth category's 5 valid products

# Cameras and Camcorders category: FotoSnap DSLR Camera, ActionCam 4K, FotoSnap Mirrorless Camera, ZoomMaster Camcorder, FotoSnap Instant Camera → 
#Sixth category's 5 valid products

# Only output the list of objects, with nothing else. → Final rule forcing clean, parseable output with no extra commentary

user_message_1 = f"""
 tell me about the smartx pro phone and \
 the fotosnap camera, the dslr one. \
 Also tell me about your tvs """
# user_message_1 → Variable name storing the result
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }

# tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also tell me about your tvs → The actual customer question, 
#mentioning two specific products and one general category (TVs)

# \ → Line continuation character: joins multiple code lines into one continuous string with no line breaks in the text

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message_1}{delimiter}"},  
] 
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content': system_message} → Key-value pair inserting the full category/product extraction prompt defined above
# {'role':'user' → Key-value pair marking this message as coming from the user

# 'content': f"{delimiter}{user_message_1}{delimiter}"} → Key-value pair wrapping the customer's question between #### delimiters, 
#exactly as the system prompt expects

category_and_product_response_1 = get_completion_from_messages(messages)
# category_and_product_response_1 → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages) → Argument passing the conversation list defined above

print(category_and_product_response_1)
# print → Built-in function that displays output
# (category_and_product_response_1) → Argument specifying what to display — the extracted list of categories/products as structured text

# This cell tests structured information extraction: it asks about the SmartX ProPhone, the FotoSnap DSLR Camera, and TVs in general, 
# and the system prompt instructs the model to output ONLY a Python-list-style structure identifying the specific products and/or category mentioned, 
# matched strictly against the allowed catalog, with no extra commentary.

[{'products': ['SmartX ProPhone', 'FotoSnap DSLR Camera']}, {'category': 'Televisions and Home Theater Systems'}]


In [5]:
#4.TESTING THE EXTRACTION SYSTEM WITH AN UNRELATED/OUT-OF-CATALOG QUERY

user_message_2 = f"""
my router isn't working"""
# user_message_2 → Variable name storing the result
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }

# my router isn't working → The actual customer message, which mentions a product (router) that is NOT in the allowed catalog 
#(no router category or product exists)

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message_2}{delimiter}"},  
] 
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content': system_message} → Key-value pair inserting the full category/product extraction prompt defined earlier
# {'role':'user' → Key-value pair marking this message as coming from the user

# 'content': f"{delimiter}{user_message_2}{delimiter}"} → Key-value pair wrapping the customer's message between #### delimiters, 
#exactly as the system prompt expects

response = get_completion_from_messages(messages)
# response → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages) → Argument passing the conversation list defined above

print(response)
# print → Built-in function that displays output
# (response) → Argument specifying what to display — the model's extraction result

# This cell tests the extraction system with a message about a product outside the allowed catalog (a router), checking whether the model correctly 
# returns an empty list rather than hallucinating a matching category or product, since routers aren't sold in any of the 6 allowed categories.

[]


In [6]:
#5.PRODUCT CATALOG DATA (FULL PRODUCT DICTIONARY)

# product information
products = {
    "TechPro Ultrabook": {
        "name": "TechPro Ultrabook",
        "category": "Computers and Laptops",
        "brand": "TechPro",
        "model_number": "TP-UB100",
        "warranty": "1 year",
        "rating": 4.5,
        "features": ["13.3-inch display", "8GB RAM", "256GB SSD", "Intel Core i5 processor"],
        "description": "A sleek and lightweight ultrabook for everyday use.",
        "price": 799.99
    },
    "BlueWave Gaming Laptop": {
        "name": "BlueWave Gaming Laptop",
        "category": "Computers and Laptops",
        "brand": "BlueWave",
        "model_number": "BW-GL200",
        "warranty": "2 years",
        "rating": 4.7,
        "features": ["15.6-inch display", "16GB RAM", "512GB SSD", "NVIDIA GeForce RTX 3060"],
        "description": "A high-performance gaming laptop for an immersive experience.",
        "price": 1199.99
    },
    "PowerLite Convertible": {
        "name": "PowerLite Convertible",
        "category": "Computers and Laptops",
        "brand": "PowerLite",
        "model_number": "PL-CV300",
        "warranty": "1 year",
        "rating": 4.3,
        "features": ["14-inch touchscreen", "8GB RAM", "256GB SSD", "360-degree hinge"],
        "description": "A versatile convertible laptop with a responsive touchscreen.",
        "price": 699.99
    },
    "TechPro Desktop": {
        "name": "TechPro Desktop",
        "category": "Computers and Laptops",
        "brand": "TechPro",
        "model_number": "TP-DT500",
        "warranty": "1 year",
        "rating": 4.4,
        "features": ["Intel Core i7 processor", "16GB RAM", "1TB HDD", "NVIDIA GeForce GTX 1660"],
        "description": "A powerful desktop computer for work and play.",
        "price": 999.99
    },
    "BlueWave Chromebook": {
        "name": "BlueWave Chromebook",
        "category": "Computers and Laptops",
        "brand": "BlueWave",
        "model_number": "BW-CB100",
        "warranty": "1 year",
        "rating": 4.1,
        "features": ["11.6-inch display", "4GB RAM", "32GB eMMC", "Chrome OS"],
        "description": "A compact and affordable Chromebook for everyday tasks.",
        "price": 249.99
    },
    "SmartX ProPhone": {
        "name": "SmartX ProPhone",
        "category": "Smartphones and Accessories",
        "brand": "SmartX",
        "model_number": "SX-PP10",
        "warranty": "1 year",
        "rating": 4.6,
        "features": ["6.1-inch display", "128GB storage", "12MP dual camera", "5G"],
        "description": "A powerful smartphone with advanced camera features.",
        "price": 899.99
    },
    "MobiTech PowerCase": {
        "name": "MobiTech PowerCase",
        "category": "Smartphones and Accessories",
        "brand": "MobiTech",
        "model_number": "MT-PC20",
        "warranty": "1 year",
        "rating": 4.3,
        "features": ["5000mAh battery", "Wireless charging", "Compatible with SmartX ProPhone"],
        "description": "A protective case with built-in battery for extended usage.",
        "price": 59.99
    },
    "SmartX MiniPhone": {
        "name": "SmartX MiniPhone",
        "category": "Smartphones and Accessories",
        "brand": "SmartX",
        "model_number": "SX-MP5",
        "warranty": "1 year",
        "rating": 4.2,
        "features": ["4.7-inch display", "64GB storage", "8MP camera", "4G"],
        "description": "A compact and affordable smartphone for basic tasks.",
        "price": 399.99
    },
    "MobiTech Wireless Charger": {
        "name": "MobiTech Wireless Charger",
        "category": "Smartphones and Accessories",
        "brand": "MobiTech",
        "model_number": "MT-WC10",
        "warranty": "1 year",
        "rating": 4.5,
        "features": ["10W fast charging", "Qi-compatible", "LED indicator", "Compact design"],
        "description": "A convenient wireless charger for a clutter-free workspace.",
        "price": 29.99
    },
    "SmartX EarBuds": {
        "name": "SmartX EarBuds",
        "category": "Smartphones and Accessories",
        "brand": "SmartX",
        "model_number": "SX-EB20",
        "warranty": "1 year",
        "rating": 4.4,
        "features": ["True wireless", "Bluetooth 5.0", "Touch controls", "24-hour battery life"],
        "description": "Experience true wireless freedom with these comfortable earbuds.",
        "price": 99.99
    },

    "CineView 4K TV": {
        "name": "CineView 4K TV",
        "category": "Televisions and Home Theater Systems",
        "brand": "CineView",
        "model_number": "CV-4K55",
        "warranty": "2 years",
        "rating": 4.8,
        "features": ["55-inch display", "4K resolution", "HDR", "Smart TV"],
        "description": "A stunning 4K TV with vibrant colors and smart features.",
        "price": 599.99
    },
    "SoundMax Home Theater": {
        "name": "SoundMax Home Theater",
        "category": "Televisions and Home Theater Systems",
        "brand": "SoundMax",
        "model_number": "SM-HT100",
        "warranty": "1 year",
        "rating": 4.4,
        "features": ["5.1 channel", "1000W output", "Wireless subwoofer", "Bluetooth"],
        "description": "A powerful home theater system for an immersive audio experience.",
        "price": 399.99
    },
    "CineView 8K TV": {
        "name": "CineView 8K TV",
        "category": "Televisions and Home Theater Systems",
        "brand": "CineView",
        "model_number": "CV-8K65",
        "warranty": "2 years",
        "rating": 4.9,
        "features": ["65-inch display", "8K resolution", "HDR", "Smart TV"],
        "description": "Experience the future of television with this stunning 8K TV.",
        "price": 2999.99
    },
    "SoundMax Soundbar": {
        "name": "SoundMax Soundbar",
        "category": "Televisions and Home Theater Systems",
        "brand": "SoundMax",
        "model_number": "SM-SB50",
        "warranty": "1 year",
        "rating": 4.3,
        "features": ["2.1 channel", "300W output", "Wireless subwoofer", "Bluetooth"],
        "description": "Upgrade your TV's audio with this sleek and powerful soundbar.",
        "price": 199.99
    },
    "CineView OLED TV": {
        "name": "CineView OLED TV",
        "category": "Televisions and Home Theater Systems",
        "brand": "CineView",
        "model_number": "CV-OLED55",
        "warranty": "2 years",
        "rating": 4.7,
        "features": ["55-inch display", "4K resolution", "HDR", "Smart TV"],
        "description": "Experience true blacks and vibrant colors with this OLED TV.",
        "price": 1499.99
    },

    "GameSphere X": {
        "name": "GameSphere X",
        "category": "Gaming Consoles and Accessories",
        "brand": "GameSphere",
        "model_number": "GS-X",
        "warranty": "1 year",
        "rating": 4.9,
        "features": ["4K gaming", "1TB storage", "Backward compatibility", "Online multiplayer"],
        "description": "A next-generation gaming console for the ultimate gaming experience.",
        "price": 499.99
    },
    "ProGamer Controller": {
        "name": "ProGamer Controller",
        "category": "Gaming Consoles and Accessories",
        "brand": "ProGamer",
        "model_number": "PG-C100",
        "warranty": "1 year",
        "rating": 4.2,
        "features": ["Ergonomic design", "Customizable buttons", "Wireless", "Rechargeable battery"],
        "description": "A high-quality gaming controller for precision and comfort.",
        "price": 59.99
    },
    "GameSphere Y": {
        "name": "GameSphere Y",
        "category": "Gaming Consoles and Accessories",
        "brand": "GameSphere",
        "model_number": "GS-Y",
        "warranty": "1 year",
        "rating": 4.8,
        "features": ["4K gaming", "500GB storage", "Backward compatibility", "Online multiplayer"],
        "description": "A compact gaming console with powerful performance.",
        "price": 399.99
    },
    "ProGamer Racing Wheel": {
        "name": "ProGamer Racing Wheel",
        "category": "Gaming Consoles and Accessories",
        "brand": "ProGamer",
        "model_number": "PG-RW200",
        "warranty": "1 year",
        "rating": 4.5,
        "features": ["Force feedback", "Adjustable pedals", "Paddle shifters", "Compatible with GameSphere X"],
        "description": "Enhance your racing games with this realistic racing wheel.",
        "price": 249.99
    },
    "GameSphere VR Headset": {
        "name": "GameSphere VR Headset",
        "category": "Gaming Consoles and Accessories",
        "brand": "GameSphere",
        "model_number": "GS-VR",
        "warranty": "1 year",
        "rating": 4.6,
        "features": ["Immersive VR experience", "Built-in headphones", "Adjustable headband", "Compatible with GameSphere X"],
        "description": "Step into the world of virtual reality with this comfortable VR headset.",
        "price": 299.99
    },

    "AudioPhonic Noise-Canceling Headphones": {
        "name": "AudioPhonic Noise-Canceling Headphones",
        "category": "Audio Equipment",
        "brand": "AudioPhonic",
        "model_number": "AP-NC100",
        "warranty": "1 year",
        "rating": 4.6,
        "features": ["Active noise-canceling", "Bluetooth", "20-hour battery life", "Comfortable fit"],
        "description": "Experience immersive sound with these noise-canceling headphones.",
        "price": 199.99
    },
    "WaveSound Bluetooth Speaker": {
        "name": "WaveSound Bluetooth Speaker",
        "category": "Audio Equipment",
        "brand": "WaveSound",
        "model_number": "WS-BS50",
        "warranty": "1 year",
        "rating": 4.5,
        "features": ["Portable", "10-hour battery life", "Water-resistant", "Built-in microphone"],
        "description": "A compact and versatile Bluetooth speaker for music on the go.",
        "price": 49.99
    },
    "AudioPhonic True Wireless Earbuds": {
        "name": "AudioPhonic True Wireless Earbuds",
        "category": "Audio Equipment",
        "brand": "AudioPhonic",
        "model_number": "AP-TW20",
        "warranty": "1 year",
        "rating": 4.4,
        "features": ["True wireless", "Bluetooth 5.0", "Touch controls", "18-hour battery life"],
        "description": "Enjoy music without wires with these comfortable true wireless earbuds.",
        "price": 79.99
    },
    "WaveSound Soundbar": {
        "name": "WaveSound Soundbar",
        "category": "Audio Equipment",
        "brand": "WaveSound",
        "model_number": "WS-SB40",
        "warranty": "1 year",
        "rating": 4.3,
        "features": ["2.0 channel", "80W output", "Bluetooth", "Wall-mountable"],
        "description": "Upgrade your TV's audio with this slim and powerful soundbar.",
        "price": 99.99
    },
    "AudioPhonic Turntable": {
        "name": "AudioPhonic Turntable",
        "category": "Audio Equipment",
        "brand": "AudioPhonic",
        "model_number": "AP-TT10",
        "warranty": "1 year",
        "rating": 4.2,
        "features": ["3-speed", "Built-in speakers", "Bluetooth", "USB recording"],
        "description": "Rediscover your vinyl collection with this modern turntable.",
        "price": 149.99
    },

    "FotoSnap DSLR Camera": {
        "name": "FotoSnap DSLR Camera",
        "category": "Cameras and Camcorders",
        "brand": "FotoSnap",
        "model_number": "FS-DSLR200",
        "warranty": "1 year",
        "rating": 4.7,
        "features": ["24.2MP sensor", "1080p video", "3-inch LCD", "Interchangeable lenses"],
        "description": "Capture stunning photos and videos with this versatile DSLR camera.",
        "price": 599.99
    },
    "ActionCam 4K": {
        "name": "ActionCam 4K",
        "category": "Cameras and Camcorders",
        "brand": "ActionCam",
        "model_number": "AC-4K",
        "warranty": "1 year",
        "rating": 4.4,
        "features": ["4K video", "Waterproof", "Image stabilization", "Wi-Fi"],
        "description": "Record your adventures with this rugged and compact 4K action camera.",
        "price": 299.99
    },
    "FotoSnap Mirrorless Camera": {
        "name": "FotoSnap Mirrorless Camera",
        "category": "Cameras and Camcorders",
        "brand": "FotoSnap",
        "model_number": "FS-ML100",
        "warranty": "1 year",
        "rating": 4.6,
        "features": ["20.1MP sensor", "4K video", "3-inch touchscreen", "Interchangeable lenses"],
        "description": "A compact and lightweight mirrorless camera with advanced features.",
        "price": 799.99
    },
    "ZoomMaster Camcorder": {
        "name": "ZoomMaster Camcorder",
        "category": "Cameras and Camcorders",
        "brand": "ZoomMaster",
        "model_number": "ZM-CM50",
        "warranty": "1 year",
        "rating": 4.3,
        "features": ["1080p video", "30x optical zoom", "3-inch LCD", "Image stabilization"],
        "description": "Capture life's moments with this easy-to-use camcorder.",
        "price": 249.99
    },
    "FotoSnap Instant Camera": {
        "name": "FotoSnap Instant Camera",
        "category": "Cameras and Camcorders",
        "brand": "FotoSnap",
        "model_number": "FS-IC10",
        "warranty": "1 year",
        "rating": 4.1,
        "features": ["Instant prints", "Built-in flash", "Selfie mirror", "Battery-powered"],
        "description": "Create instant memories with this fun and portable instant camera.",
        "price": 69.99
    }
}

In [7]:
#6.HELPER FUNCTIONS TO LOOK UP PRODUCTS FROM THE CATALOG

def get_product_by_name(name):
# def → Keyword used to define a function
# get_product_by_name → Name given to the function
# (name) → Parameter representing the exact product name to search for
# : → Marks the start of the function body

    return products.get(name, None)
    # return → Keyword that sends a value back from the function
    # products → The product catalog dictionary defined earlier
    # .get → Function that looks up a key and returns a default value if the key is not found
    # (name → Argument specifying the key (product name) to search for
    # None) → Argument specifying the default value to return if the product isn't found (instead of raising an error)

def get_products_by_category(category):
# def → Keyword used to define a function
# get_products_by_category → Name given to the function
# (category) → Parameter representing the category name to filter by
# : → Marks the start of the function body

    return [product for product in products.values() if product["category"] == category]
    # return → Keyword that sends a value back from the function
    # [ ] → List comprehension brackets, building a new list
    # product → The item being collected into the list (each matching product dictionary)
    # for → Keyword starting the loop inside the list comprehension
    # product → Loop variable representing each product dictionary
    # in → Keyword linking the loop variable to the iterable
    # products.values() → Returns all the product detail dictionaries (without their keys/names)
    # if → Keyword adding a filter condition to the list comprehension
    # product["category"] → Looks up the "category" key inside each product's dictionary
    # == → Equality comparison operator
    # category → The category name passed into the function

# These two helper functions let you look up catalog data programmatically instead of relying on the model's memory: get_product_by_name finds 
# one product by its exact name (returning None if not found), and get_products_by_category returns a list of all products belonging to a given 
# category.

In [8]:
#7.TESTING get_product_by_name

print(get_product_by_name("TechPro Ultrabook"))
# print → Built-in function that displays output
# get_product_by_name → The helper function defined earlier that looks up a product by its exact name
# ("TechPro Ultrabook") → Argument passing the exact product name to search for in the catalog

# This cell tests get_product_by_name by looking up "TechPro Ultrabook" and printing its full detail dictionary 
# (name, category, brand, model_number, warranty, rating, features, description, price).

{'name': 'TechPro Ultrabook', 'category': 'Computers and Laptops', 'brand': 'TechPro', 'model_number': 'TP-UB100', 'warranty': '1 year', 'rating': 4.5, 'features': ['13.3-inch display', '8GB RAM', '256GB SSD', 'Intel Core i5 processor'], 'description': 'A sleek and lightweight ultrabook for everyday use.', 'price': 799.99}


In [9]:
#8.TESTING get_products_by_category

print(get_products_by_category("Computers and Laptops"))
# print → Built-in function that displays output
# get_products_by_category → The helper function defined earlier that filters products by category
# ("Computers and Laptops") → Argument passing the category name to filter by

# This cell tests get_products_by_category by requesting all products in the "Computers and Laptops" category, 
# and prints the resulting list of 5 matching product dictionaries (TechPro Ultrabook, BlueWave Gaming Laptop, PowerLite Convertible, 
# TechPro Desktop, BlueWave Chromebook).

[{'name': 'TechPro Ultrabook', 'category': 'Computers and Laptops', 'brand': 'TechPro', 'model_number': 'TP-UB100', 'warranty': '1 year', 'rating': 4.5, 'features': ['13.3-inch display', '8GB RAM', '256GB SSD', 'Intel Core i5 processor'], 'description': 'A sleek and lightweight ultrabook for everyday use.', 'price': 799.99}, {'name': 'BlueWave Gaming Laptop', 'category': 'Computers and Laptops', 'brand': 'BlueWave', 'model_number': 'BW-GL200', 'warranty': '2 years', 'rating': 4.7, 'features': ['15.6-inch display', '16GB RAM', '512GB SSD', 'NVIDIA GeForce RTX 3060'], 'description': 'A high-performance gaming laptop for an immersive experience.', 'price': 1199.99}, {'name': 'PowerLite Convertible', 'category': 'Computers and Laptops', 'brand': 'PowerLite', 'model_number': 'PL-CV300', 'warranty': '1 year', 'rating': 4.3, 'features': ['14-inch touchscreen', '8GB RAM', '256GB SSD', '360-degree hinge'], 'description': 'A versatile convertible laptop with a responsive touchscreen.', 'price': 

In [10]:
#9.PRINTING THE EARLIER STORED USER MESSAGE

print(user_message_1)
# print → Built-in function that displays output
# (user_message_1) → Argument specifying what to display — the customer question stored earlier ("tell me about the smartx pro phone and the 
# fotosnap camera, the dslr one. Also tell me about your tvs")

# This cell simply prints the user_message_1 variable defined earlier, showing the original customer query text.


 tell me about the smartx pro phone and  the fotosnap camera, the dslr one.  Also tell me about your tvs 


In [11]:
#10.PRINTING THE EXTRACTED CATEGORY/PRODUCT RESPONSE

print(category_and_product_response_1)
# print → Built-in function that displays output
# (category_and_product_response_1) → Argument specifying what to display — the model's structured extraction output stored earlier, 
# identifying the categories/products mentioned in user_message_1

# This cell prints category_and_product_response_1, showing the Python-list-style structure the model extracted from the SmartX ProPhone / 
# FotoSnap DSLR / TVs query.

[{'products': ['SmartX ProPhone', 'FotoSnap DSLR Camera']}, {'category': 'Televisions and Home Theater Systems'}]


In [12]:
# Read Python string into Python list of dictionaries

#11.FUNCTION TO SAFELY CONVERT THE MODEL'S STRING OUTPUT INTO A PYTHON LIST

import json
# import → Keyword used to bring a library into the notebook
# json → Built-in library for working with JSON data (converting between JSON text and Python objects)

def read_string_to_list(input_string):
# def → Keyword used to define a function
# read_string_to_list → Name given to the function
# (input_string) → Parameter representing the raw text string to convert (e.g. the model's response)
# : → Marks the start of the function body

    if input_string is None:
    # if → Keyword starting a conditional check
    # input_string → The text passed into the function
    # is None → Checks whether the value is exactly None (nothing provided)
    # : → Marks the start of the if-block

        return None
        # return → Keyword that sends a value back from the function
        # None → Returned immediately if there's no input to process

    try:
    # try → Keyword starting a block that may raise an error

        input_string = input_string.replace("'", "\"")  # Replace single quotes with double quotes for valid JSON
        # input_string → Variable being reassigned
        # = → Assignment operator
        # input_string → The original text
        # .replace → Function that finds and swaps characters in a string
        # ("'", "\"") → Arguments: replace every single quote ' with an escaped double quote "
        
        # # Replace single quotes with double quotes for valid JSON → Comment explaining why this step is needed (the model may output Python-style 
        # quotes, but JSON requires double quotes)

        data = json.loads(input_string)
        # data → Variable name storing the result
        # = → Assignment operator
        # json → The json library
        # .loads → Function that parses a JSON-formatted string and converts it into a Python object (list/dict)
        # (input_string) → Argument passing the cleaned-up string to parse

        return data
        # return → Keyword that sends a value back from the function
        # data → The successfully parsed Python list/dictionary

    except json.JSONDecodeError:
    # except → Keyword catching a specific error
    # json.JSONDecodeError → The error type raised when the text isn't valid JSON (e.g. malformed syntax)
    # : → Marks the start of the except-block

        print("Error: Invalid JSON string")
        # print → Built-in function that displays output
        # ("Error: Invalid JSON string") → Argument specifying the error message shown when parsing fails

        return None
        # return → Keyword that sends a value back from the function
        # None → Returned as a safe fallback when the conversion fails

# This function safely converts the model's text output (like category_and_product_response_1) into an actual usable Python list/dictionary: 
# it handles missing input, fixes single-quote formatting so it matches valid JSON syntax, and catches parsing errors instead of crashing the program.

In [13]:
#12.CONVERTING THE MODEL'S TEXT RESPONSE INTO AN ACTUAL PYTHON LIST

category_and_product_list = read_string_to_list(category_and_product_response_1)
# category_and_product_list → Variable name storing the result
# = → Assignment operator
# read_string_to_list → The function defined earlier that safely converts a text string into a Python list/dictionary
# (category_and_product_response_1) → Argument passing the model's raw text output (from the earlier extraction cell) to be parsed

print(category_and_product_list)
# print → Built-in function that displays output
# (category_and_product_list) → Argument specifying what to display — the parsed Python list, now usable in code (e.g. for looping, indexing, 
# or passing to get_product_by_name/get_products_by_category)

# This cell converts category_and_product_response_1 (plain text from the model) into a real Python list using read_string_to_list, 
# then prints it — this is the step that turns the model's text output into structured data your program can actually work with.

[{'products': ['SmartX ProPhone', 'FotoSnap DSLR Camera']}, {'category': 'Televisions and Home Theater Systems'}]


In [14]:
# Retrieve detailed product information for the relevant products and categories

#13.FUNCTION TO BUILD A FULL PRODUCT DETAIL STRING FROM THE EXTRACTED LIST

def generate_output_string(data_list):
# def → Keyword used to define a function
# generate_output_string → Name given to the function
# (data_list) → Parameter representing the parsed category/product list (e.g. category_and_product_list)
# : → Marks the start of the function body

    output_string = ""
    # output_string → Variable name storing the result
    # = → Assignment operator
    # "" → Empty string that will be built up piece by piece as product details are added

    if data_list is None:
    # if → Keyword starting a conditional check
    # data_list → The list passed into the function
    # is None → Checks whether the value is exactly None (nothing to process)
    # : → Marks the start of the if-block

        return output_string
        # return → Keyword that sends a value back from the function
        # output_string → The still-empty string, returned immediately if there's nothing to process

    for data in data_list:
    # for → Keyword starting a loop
    # data → Loop variable representing each item (dictionary) in the list
    # in → Keyword linking the loop variable to the iterable
    # data_list → The list being looped over
    # : → Marks the start of the loop body

        try:
        # try → Keyword starting a block that may raise an error

            if "products" in data:
            # if → Keyword starting a conditional check
            # "products" → The key being searched for
            # in → Membership operator checking whether the key exists
            # data → The current dictionary item
            # : → Marks the start of the if-block

                products_list = data["products"]
                # products_list → Variable name storing the result
                # = → Assignment operator
                # data["products"] → Looks up the "products" key, which holds a list of specific product names

                for product_name in products_list:
                # for → Keyword starting a loop
                # product_name → Loop variable representing each product name string
                # in → Keyword linking the loop variable to the iterable
                # products_list → The list of product names being looped over
                # : → Marks the start of the loop body

                    product = get_product_by_name(product_name)
                    # product → Variable name storing the result
                    # = → Assignment operator
                    # get_product_by_name → The helper function defined earlier that looks up a product by its exact name
                    # (product_name) → Argument passing the current product name to search for

                    if product:
                    # if → Keyword starting a conditional check
                    # product → Checks whether a matching product was found (truthy if not None)
                    # : → Marks the start of the if-block

                        output_string += json.dumps(product, indent=4) + "\n"
                        # output_string → The string being modified
                        # += → Addition-assignment operator, appending to the existing string
                        # json → The json library
                        # .dumps → Function that converts a Python object into a formatted JSON string
                        # (product → Argument passing the product's detail dictionary
                        # indent=4) → Argument formatting the JSON with 4-space indentation for readability
                        # + "\n" → Adds a newline after each product's details

                    else:
                    # else → Keyword defining the alternative branch when no product was found

                        print(f"Error: Product '{product_name}' not found")
                        # print → Built-in function that displays output
                        # (f"Error: Product '{product_name}' not found") → f-string showing which product name couldn't be matched in the catalog

            elif "category" in data:
            # elif → Keyword for an additional condition, checked only if the first "if" was False
            # "category" → The key being searched for
            # in → Membership operator checking whether the key exists
            # data → The current dictionary item
            # : → Marks the start of the elif-block

                category_name = data["category"]
                # category_name → Variable name storing the result
                # = → Assignment operator
                # data["category"] → Looks up the "category" key, holding the category name string

                category_products = get_products_by_category(category_name)
                # category_products → Variable name storing the result
                # = → Assignment operator
                # get_products_by_category → The helper function defined earlier that filters products by category
                # (category_name) → Argument passing the category name to filter by

                for product in category_products:
                # for → Keyword starting a loop
                # product → Loop variable representing each matching product dictionary
                # in → Keyword linking the loop variable to the iterable
                # category_products → The list of products in that category
                # : → Marks the start of the loop body

                    output_string += json.dumps(product, indent=4) + "\n"
                    # output_string → The string being modified
                    # += → Addition-assignment operator, appending to the existing string
                    # json.dumps(product, indent=4) → Converts this product's details into a formatted JSON string
                    # + "\n" → Adds a newline after each product's details

            else:
            # else → Keyword defining the fallback branch when neither "products" nor "category" is present

                print("Error: Invalid object format")
                # print → Built-in function that displays output
                # ("Error: Invalid object format") → Argument specifying the error message for an unrecognized data structure

        except Exception as e:
        # except → Keyword catching an error if the try block fails
        # Exception → The general error type being caught (catches any kind of error)
        # as e → Stores the error details in a variable named e

            print(f"Error: {e}")
            # print → Built-in function that displays output
            # (f"Error: {e}") → f-string showing the actual error message that occurred

    return output_string
    # return → Keyword that sends a value back from the function
    # output_string → The fully built string containing all matched products' JSON details, separated by newlines

# This function takes the parsed category/product list and builds a single text string containing the full JSON details of every matched product: 
# it looks up specific products by name when a "products" key is found, or looks up all products in a category when a "category" key is found, 
# skipping and logging any products/categories that can't be matched, and safely catching any unexpected errors along the way.

In [15]:
#14.GENERATING THE FULL PRODUCT DETAIL STRING FOR THE USER'S QUERY

product_information_for_user_message_1 = generate_output_string(category_and_product_list)
# product_information_for_user_message_1 → Variable name storing the result
# = → Assignment operator
# generate_output_string → The function defined earlier that builds a full JSON-formatted string of product details

# (category_and_product_list) → Argument passing the parsed list (from the extraction + JSON-parsing steps) containing the categories/
#products mentioned in user_message_1

print(product_information_for_user_message_1)
# print → Built-in function that displays output
# (product_information_for_user_message_1) → Argument specifying what to display — the full, detailed product information (as formatted JSON text) 
# for every product matched from the customer's original query

# This cell runs the full retrieval pipeline end-to-end: it takes the extracted category/product list and looks up the REAL catalog data for 
# each match, building a complete, accurate product information string. This grounds the model's final answer in actual data instead of letting it 
# hallucinate specs from memory.

{
    "name": "SmartX ProPhone",
    "category": "Smartphones and Accessories",
    "brand": "SmartX",
    "model_number": "SX-PP10",
    "warranty": "1 year",
    "rating": 4.6,
    "features": [
        "6.1-inch display",
        "128GB storage",
        "12MP dual camera",
        "5G"
    ],
    "description": "A powerful smartphone with advanced camera features.",
    "price": 899.99
}
{
    "name": "FotoSnap DSLR Camera",
    "category": "Cameras and Camcorders",
    "brand": "FotoSnap",
    "model_number": "FS-DSLR200",
    "warranty": "1 year",
    "rating": 4.7,
    "features": [
        "24.2MP sensor",
        "1080p video",
        "3-inch LCD",
        "Interchangeable lenses"
    ],
    "description": "Capture stunning photos and videos with this versatile DSLR camera.",
    "price": 599.99
}
{
    "name": "CineView 4K TV",
    "category": "Televisions and Home Theater Systems",
    "brand": "CineView",
    "model_number": "CV-4K55",
    "warranty": "2 years",
    "ratin

In [16]:
# Generate answer to user query based on detailed product information

#15.GENERATING THE FINAL CUSTOMER-FACING RESPONSE USING RETRIEVED PRODUCT DATA

system_message = f"""
You are a customer service assistant for a \
large electronic store. \
Respond in a friendly and helpful tone, \
with very concise answers. \
Make sure to ask the user relevant follow up questions.
"""
# system_message → Variable name storing the result
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }
# You are a customer service assistant for a large electronic store. → Role instruction telling the LLM who it is
# Respond in a friendly and helpful tone, with very concise answers. → Style instruction controlling tone and length
# Make sure to ask the user relevant follow up questions. → Behavior instruction encouraging continued engagement
# \ → Line continuation character: joins multiple code lines into one continuous string with no line breaks in the text

user_message_1 = f"""
tell me about the smartx pro phone and \
the fotosnap camera, the dslr one. \
Also tell me about your tvs"""
# user_message_1 → Variable name storing the result (overwrites the earlier value with the same text)
# = → Assignment operator
# f""" ... """ → Triple-quoted f-string: a multi-line string that can insert variable values inside { }
# tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also tell me about your tvs → The original customer question, 
# restated here for this new conversation

messages =  [  
{'role':'system', 
 'content': system_message},
{'role':'user', 
 'content': user_message_1},  
{'role':'assistant', 
 'content': f"""Relevant product information:\n\
 {product_information_for_user_message_1}"""},   
]
# messages → Variable name storing the result
# = → Assignment operator
# [ ] → List brackets holding the full conversation history
# {'role':'system' → Key-value pair marking this message as the system instruction
# 'content': system_message} → Key-value pair inserting the friendly customer-service persona defined above
# {'role':'user' → Key-value pair marking this message as coming from the user
# 'content': user_message_1} → Key-value pair holding the customer's original question
# {'role':'assistant' → Key-value pair marking this as a simulated prior assistant turn (a trick to inject data into context)

# 'content': f"""Relevant product information:\n {product_information_for_user_message_1}"""} → Key-value pair injecting the REAL product data 
# retrieved earlier, framed as if the assistant already looked it up — this grounds the final answer in actual catalog facts instead of the 
# model's memory

# \n → Newline character separating the label from the actual data
# \ → Line continuation character: joins the two code lines into one continuous string with no line break in the text

final_response = get_completion_from_messages(messages)
# final_response → Variable name storing the result
# = → Assignment operator
# get_completion_from_messages → The function defined earlier that sends a full message list and returns the model's reply
# (messages) → Argument passing the conversation list defined above

print(final_response)
# print → Built-in function that displays output
# (final_response) → Argument specifying what to display — the model's final, friendly, accurate response to the customer

# This cell completes the full RAG-style (Retrieval-Augmented Generation) pipeline: it sets a friendly customer-service persona, 
# restates the customer's question, injects the REAL retrieved product data as a fake "assistant" turn to ground the model in facts, 
# then generates a final concise, accurate response that answers the customer using genuine catalog data instead of hallucinated specs.

Here’s a quick overview:

### SmartX ProPhone — $899.99
- 6.1-inch display  
- 128GB storage  
- 5G connectivity  
- 12MP dual camera  
- 1-year warranty  
- Rating: 4.6/5  

### FotoSnap DSLR Camera — $599.99
- 24.2MP sensor  
- Interchangeable lenses  
- 1080p video recording  
- 3-inch LCD  
- 1-year warranty  
- Rating: 4.7/5  

### TVs
- **CineView 4K TV, 55-inch — $599.99:** HDR, Smart TV, 4K resolution  
- **CineView OLED TV, 55-inch — $1,499.99:** 4K HDR, Smart TV, deep blacks and vibrant colors  
- **CineView 8K TV, 65-inch — $2,999.99:** 8K HDR, Smart TV  

All CineView TVs include a 2-year warranty. We also offer a **SoundMax Soundbar for $199.99** and a **SoundMax 5.1-channel home theater system for $399.99**.

Are you mainly looking for the best value, the best picture quality, or a phone and camera for photography?
